# Building an environment

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

The environment is the other half of every experiment : the dish the larvae are in, the food and
odor sources placed in it, the walls they can bump into, and the sensory fields spread over it.
Change the environment and you change the assay - the same model in a dish with one odor source is
a chemotaxis experiment, and in an empty dish it is a free-exploration experiment.

This notebook builds environments : first by reading the ones that ship with the package, then by
assembling one field at a time, and finally with the shortcuts that make the common layouts
one-liners.

**What you will be able to do afterwards**

- Read any stored environment configuration and say what each field controls.
- Set arena geometry, dimensions and toroidal wrap-around.
- Place individual food and odor sources, groups of sources, and a uniform food grid.
- Add borders as obstacles.
- Use the factory shortcuts (`dish`, `rect`, `maze`, `double_patch`, ...) instead of writing the
  configuration by hand.
- Render an environment to check it before running anything in it.

**Prerequisites** :
[The configuration registry](configuration_registry.ipynb).

**Cost** : seconds, unless you turn the rendering on.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_VISUAL_DEMO` | `False` | rendering environments in a pygame window |
| `RUN_WRITE_DEMO` | `False` | storing the environment built here in the registry |

## Setup

In [1]:
%matplotlib inline

%load_ext param.ipython

import larvaworld
from larvaworld.lib import reg
from larvaworld.lib.param.composition import Odor
from larvaworld.lib.reg.generators import EnvConf, FoodConf

larvaworld.VERBOSE = 1

# Tutorial safety switches
RUN_VISUAL_DEMO = False  # opens a pygame window
RUN_WRITE_DEMO = False  # writes to the on-disc conf dict

Welcome to the param IPython extension! (https://param.holoviz.org/)
Available magics: %params


Initializing larvaworld registry


Registry configured!


## Section 1 : What an environment is made of

`EnvConf` has six fields, and every stored environment is some combination of them :

| field | what it is | required |
|---|---|---|
| `arena` | shape, size and topology of the space | yes |
| `food_params` | the food and odor sources in it | yes (may be empty) |
| `border_list` | walls and obstacles | no |
| `odorscape` | how odor concentration is computed over space | only if there are odors |
| `windscape` | wind direction, speed and air puffs | no |
| `thermoscape` | the temperature field | no |

The three `*scape` fields default to `None`, which is the sensible default : a landscape costs
computation at every timestep, so it only exists if the assay needs it.

In [2]:
%params EnvConf

The environments that ship with the package cover the standard assays. Their IDs are descriptive,
and reading a couple of them is the fastest way to learn the vocabulary.

In [3]:
envIDs = reg.conf.Env.confIDs
print(f"{len(envIDs)} stored environments :")
print(envIDs)

35 stored environments :
['4corners', 'CS_UCS_off_food', 'CS_UCS_on_food', 'CS_UCS_on_food_x2', 'arena_1000mm', 'arena_200mm', 'arena_500mm', 'arena_50mm_diffusion', 'dish', 'dish_40mm', 'double_patch', 'focus', 'food_at_bottom', 'food_grid', 'game', 'maze', 'mid_odor_diffusion', 'mid_odor_gaussian', 'multi_patch', 'odor_gaussian_square', 'odor_gradient', 'odors_sources_in_oval', 'patch_grid', 'patchy_food', 'puff_arena_bordered', 'random_food', 'single_odor_patch', 'single_patch', 'single_puff', 'test_group_odor_arena', 'thermo_arena', 'uniform_food', 'windy_arena', 'windy_arena_bordered', 'windy_blob_arena']


In [4]:
envID = envIDs[0]
env_entry = reg.conf.Env.getID(envID)

print(f"Environment {envID!r} :")
env_entry.print()

Environment '4corners' :
     arena : 
          dims : (0.2, 0.2)
          geometry : rectangular
          torus : False
     border_list : 
     food_params : 
          food_grid : None
          source_groups : 
          source_units : 
               Source_0 : 
                    amount : 0.01
                    can_be_carried : False
                    can_be_displaced : False
                    color : blue
                    group : None
                    odor : 
                         id : Odor_0
                         intensity : 2.0
                         spread : 0.01
                    pos : (-0.05, -0.05)
                    radius : 0.01
                    regeneration : False
                    regeneration_pos : None
                    substrate : 
                         composition : 
                              glucose : 0.0
                              dextrose : 0.0
                              saccharose : 0.0
                           

## Section 2 : The arena

The arena is an `Area` : two dimensions in **metres**, a geometry, and an optional toroidal
topology in which an agent leaving one edge reappears at the opposite one.

Sizes are in metres throughout Larvaworld, so a 10 cm petri dish is `dims=(0.1, 0.1)`. This trips
up everybody once.

In [5]:
env = EnvConf()

print("Default arena :")
print(f"  dims     = {env.arena.dims} (metres)")
print(f"  geometry = {env.arena.geometry}")
print(f"  torus    = {env.arena.torus}")

Default arena :
  dims     = (0.1, 0.1) (metres)
  geometry = circular
  torus    = False


In [6]:
# A circular 15 cm dish
env.arena.geometry = "circular"
env.arena.dims = (0.15, 0.15)

print(f"{env.arena.geometry} arena of {env.arena.dims} m")

circular arena of (0.15, 0.15) m


## Section 3 : Food and odor sources

`food_params` holds three different ways of putting something edible or smellable in the arena :

- **`source_units`** - individual sources, each with its own position, radius, amount and odor.
  This is what you want for *one odor source in the middle of the dish*.
- **`source_groups`** - a group of sources placed by a spatial distribution, for scattered patches.
- **`food_grid`** - a uniform layer of food covering the whole arena, for free-feeding assays.

A source carries an `Odor` if it should be smelled. `Odor.oG` builds a Gaussian odor and `Odor.oD`
a diffusion one; the choice has to match the `odorscape` used by the environment, which is what the
next notebook is about.

In [7]:
source = reg.gen.Food(
    unique_id="Source",
    group="Source",
    pos=(0.0, 0.0),
    radius=0.003,
    amount=0.0,
    odor=Odor.oG(id="Odor"),
    c="cyan",
)

env.food_params = FoodConf(source_units=source.entry())

print("Sources in the arena :", env.food_params.source_units.keylist)
print()
print(env.food_params.source_units.Source.odor)

Sources in the arena : ['Source']

<Odor Odor00132>


## Section 4 : Borders

A border is a polyline the agents cannot cross - a wall of a maze, a barrier between two halves of
an arena, the rim of a channel. Each has vertices in metres and a width.

In [8]:
env.border_list = {
    "wall": reg.gen.Border(vertices=[(-0.03, 0.02), (0.03, 0.02)]),
}

print("Borders :", list(env.border_list.keys()))

Borders : ['wall']


## Section 5 : The shortcuts

Assembling an environment field by field is useful for understanding it, and rarely what you want
to write. `EnvConf` and `FoodConf` provide class methods for the layouts that keep recurring :

| shortcut | the arena it builds |
|---|---|
| `EnvConf.dish(xy)` | a circular dish of side `xy` metres |
| `EnvConf.rect(xy)` | a rectangular arena; `xy` may be a float or a `(w, h)` tuple |
| `EnvConf.maze(n, h)` | a rectangular arena filled with a generated maze |
| `EnvConf.odor_gradient(dim)` | a rectangle with one odor source, for chemotaxis |
| `EnvConf.double_patch(dim)` | two food patches, for patch-choice assays |
| `FoodConf.double_patch(x, r)` | just the two patches, to drop into another arena |
| `FoodConf.CS_UCS(...)` | the paired-odor layout used in learning assays |

The `o` argument of most of these selects the odorscape : `"G"` for Gaussian, `"D"` for diffusion.

In [9]:
dish = EnvConf.dish(0.1)
print("dish        :", dish.arena.geometry, dish.arena.dims)

corridor = EnvConf.rect((0.2, 0.05))
print("rect        :", corridor.arena.geometry, corridor.arena.dims)

chemo = EnvConf.odor_gradient(dim=(0.1, 0.06), o="G")
print(
    "chemotaxis  :",
    chemo.arena.dims,
    "| odorscape :",
    chemo.odorscape.__class__.__name__,
)
print("             sources :", chemo.food_params.source_units.keylist)

dish        : circular (0.1, 0.1)
rect        : rectangular (0.2, 0.05)
chemotaxis  : (0.1, 0.06) | odorscape : GaussianValueLayerUnit
             sources : ['Source']


## Section 6 : Looking at it

`visualize()` launches a simulation with no agents in it and renders the environment, which is the
quickest way to confirm that the arena, the sources and the borders are where you think they are.
It opens a pygame window, so it is off by default.

In [10]:
if RUN_VISUAL_DEMO:
    chemo.visualize(duration=0.3)
else:
    print("Set RUN_VISUAL_DEMO = True to render this environment.")

Set RUN_VISUAL_DEMO = True to render this environment.


## Section 7 : Keeping it

An environment is only useful once other configurations can refer to it, which means storing it in
the registry under an ID. From then on any experiment can use it by name.

In [11]:
if RUN_WRITE_DEMO:
    new_id = "my_chemotaxis_arena"
    reg.conf.Env.setID(id=new_id, conf=chemo.nestedConf)
    print(
        f"Stored {new_id!r}; it is now one of {len(reg.conf.Env.confIDs)} environments"
    )

    reg.conf.Env.delete(id=new_id)
    print(f"Deleted {new_id!r} again")
else:
    print("Set RUN_WRITE_DEMO = True to store this environment in the registry.")

Set RUN_WRITE_DEMO = True to store this environment in the registry.


The same job can be done in a browser : the Portal's **Environment Builder** draws arenas, sources
and borders on a canvas and writes them to the same registry. See
[Web applications](../../visualization/web_applications.md).

## Where to go next

- [Sensory landscapes](sensory_landscapes.ipynb) - the odor, temperature and wind fields that go on
  top of the arena you just built.
- [The Python API](../1_getting_started/python_api_basics.ipynb) - putting larva groups into it.
- Reference : [Arenas and substrates](../../agents_environments/arenas_and_substrates.md).